In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel('British Airways Summer Schedule Dataset - Forage Data Science Task 1.xlsx')
print("Shape: ", df.shape)
df.head(5)

Shape:  (10000, 17)


,FLIGHT_DATE,FLIGHT_TIME,TIME_OF_DAY,AIRLINE_CD,FLIGHT_NO,DEPARTURE_STATION_CD,ARRIVAL_STATION_CD,ARRIVAL_COUNTRY,ARRIVAL_REGION,HAUL,AIRCRAFT_TYPE,FIRST_CLASS_SEATS,BUSINESS_CLASS_SEATS,ECONOMY_SEATS,TIER1_ELIGIBLE_PAX,TIER2_ELIGIBLE_PAX,TIER3_ELIGIBLE_PAX
0,2025-09-02,14:19:00,Afternoon,BA,BA5211,LHR,LAX,USA,North America,LONG,B777,8,49,178,0,10,38
1,2025-06-10,06:42:00,Morning,BA,BA7282,LHR,LAX,USA,North America,LONG,B777,8,49,178,0,7,28
2,2025-10-27,15:33:00,Afternoon,BA,BA1896,LHR,FRA,Germany,Europe,SHORT,A320,0,17,163,0,11,40
3,2025-06-15,18:29:00,Evening,BA,BA5497,LHR,IST,Turkey,Europe,SHORT,A320,0,8,172,0,16,54
4,2025-08-25,20:35:00,Evening,BA,BA1493,LHR,FRA,Germany,Europe,SHORT,A320,0,13,167,0,6,27


In [3]:
df.columns

Index(['FLIGHT_DATE', 'FLIGHT_TIME', 'TIME_OF_DAY', 'AIRLINE_CD', 'FLIGHT_NO',
       'DEPARTURE_STATION_CD', 'ARRIVAL_STATION_CD', 'ARRIVAL_COUNTRY',
       'ARRIVAL_REGION', 'HAUL', 'AIRCRAFT_TYPE', 'FIRST_CLASS_SEATS',
       'BUSINESS_CLASS_SEATS', 'ECONOMY_SEATS', 'TIER1_ELIGIBLE_PAX',
       'TIER2_ELIGIBLE_PAX', 'TIER3_ELIGIBLE_PAX'],
      dtype='str')

In [4]:
df.duplicated().sum()

np.int64(0)

In [5]:
df.isnull().sum()

FLIGHT_DATE             0
FLIGHT_TIME             0
TIME_OF_DAY             0
AIRLINE_CD              0
FLIGHT_NO               0
DEPARTURE_STATION_CD    0
ARRIVAL_STATION_CD      0
ARRIVAL_COUNTRY         0
ARRIVAL_REGION          0
HAUL                    0
AIRCRAFT_TYPE           0
FIRST_CLASS_SEATS       0
BUSINESS_CLASS_SEATS    0
ECONOMY_SEATS           0
TIER1_ELIGIBLE_PAX      0
TIER2_ELIGIBLE_PAX      0
TIER3_ELIGIBLE_PAX      0
dtype: int64

In [6]:
print("HAUL values:", df['HAUL'].unique())
print("\nTime of Day values:", df['TIME_OF_DAY'].unique())

HAUL values: <StringArray>
['LONG', 'SHORT']
Length: 2, dtype: str

Time of Day values: <StringArray>
['Afternoon', 'Morning', 'Evening', 'Lunchtime']
Length: 4, dtype: str


In [7]:
# Cal total passengers per flight
df['TOTAL_PAX'] = (df['FIRST_CLASS_SEATS'] + df['BUSINESS_CLASS_SEATS'] + df['ECONOMY_SEATS'])

In [8]:
# Cal eligibility % per flight
df['TIER1_%'] = (df['TIER1_ELIGIBLE_PAX'] / df['TOTAL_PAX']) * 100
df['TIER2_%'] = (df['TIER2_ELIGIBLE_PAX'] / df['TOTAL_PAX']) * 100
df['TIER3_%'] = (df['TIER3_ELIGIBLE_PAX'] / df['TOTAL_PAX']) * 100

In [9]:
df.head()

,FLIGHT_DATE,FLIGHT_TIME,TIME_OF_DAY,AIRLINE_CD,FLIGHT_NO,DEPARTURE_STATION_CD,ARRIVAL_STATION_CD,ARRIVAL_COUNTRY,ARRIVAL_REGION,HAUL,...,FIRST_CLASS_SEATS,BUSINESS_CLASS_SEATS,ECONOMY_SEATS,TIER1_ELIGIBLE_PAX,TIER2_ELIGIBLE_PAX,TIER3_ELIGIBLE_PAX,TOTAL_PAX,TIER1_%,TIER2_%,TIER3_%
0,2025-09-02,14:19:00,Afternoon,BA,BA5211,LHR,LAX,USA,North America,LONG,...,8,49,178,0,10,38,235,0.0,4.255319,16.170213
1,2025-06-10,06:42:00,Morning,BA,BA7282,LHR,LAX,USA,North America,LONG,...,8,49,178,0,7,28,235,0.0,2.978723,11.914894
2,2025-10-27,15:33:00,Afternoon,BA,BA1896,LHR,FRA,Germany,Europe,SHORT,...,0,17,163,0,11,40,180,0.0,6.111111,22.222222
3,2025-06-15,18:29:00,Evening,BA,BA5497,LHR,IST,Turkey,Europe,SHORT,...,0,8,172,0,16,54,180,0.0,8.888889,30.000000
4,2025-08-25,20:35:00,Evening,BA,BA1493,LHR,FRA,Germany,Europe,SHORT,...,0,13,167,0,6,27,180,0.0,3.333333,15.000000


In [10]:
print(df[['TIER1_%', 'TIER2_%', 'TIER3_%']].mean().round(2))

TIER1_%     0.29
TIER2_%     3.80
TIER3_%    14.55
dtype: float64


In [11]:
#Build the lookup table
# Group by HAUL + TIME_OF_DAY, average the % columns

lookup_table = (df.groupby(['HAUL', 'TIME_OF_DAY'])
                  .agg(
                      TIER1_PCT = ('TIER1_%', 'mean'), 
                      TIER2_PCT = ('TIER2_%', 'mean'),  
                      TIER3_PCT = ('TIER3_%', 'mean'),   
                      FLIGHT_COUNT = ('FLIGHT_NO', 'count') 
                  )
                  .round(2)
                  .reset_index()
               )

print(lookup_table)

    HAUL TIME_OF_DAY  TIER1_PCT  TIER2_PCT  TIER3_PCT  FLIGHT_COUNT
0   LONG   Afternoon       0.21       2.94      11.17           939
1   LONG     Evening       0.23       2.86      10.97          1190
2   LONG   Lunchtime       0.19       2.89      11.06           435
3   LONG     Morning       0.22       2.95      11.25          1461
4  SHORT   Afternoon       0.34       4.31      16.59          1366
5  SHORT     Evening       0.33       4.45      17.00          1783
6  SHORT   Lunchtime       0.39       4.55      17.28           757
7  SHORT     Morning       0.34       4.36      16.74          2069


In [12]:
# Get real example destinations for each group
examples = (df.groupby(['HAUL', 'TIME_OF_DAY'])
              .apply(lambda x: ', '.join(
                  x['ARRIVAL_STATION_CD']
                  .sample(min(3, len(x)), random_state=42)
                  .values
              ))
              .reset_index()
           )
examples.columns = ['HAUL', 'TIME_OF_DAY', 'EXAMPLE_DESTINATIONS']
print(examples)

    HAUL TIME_OF_DAY EXAMPLE_DESTINATIONS
0   LONG   Afternoon        DFW, LAX, DXB
1   LONG     Evening        DFW, DFW, HND
2   LONG   Lunchtime        ORD, HND, HND
3   LONG     Morning        HND, LAX, HND
4  SHORT   Afternoon        FRA, CDG, VIE
5  SHORT     Evening        BCN, MUC, MAD
6  SHORT   Lunchtime        MUC, ZRH, AMS
7  SHORT     Morning        AMS, MUC, MUC
